# 🏦 Consolidación de Datos de Transacciones

**Proyecto:** Pipeline de Datos Financieros - Prueba Técnica Iris CF  
**Ingeniero:** Nelson Andrés Salinas Zapata  
**Tecnología:** PySpark (Local), Pandas. 

----

## Paso 0: Inicialización del Entorno y Carga Cruda
En esta sección encendemos el motor de PySpark configurando una sesión local que aproveche todos los hilos del procesador (`local[*]`). Cargaremos las fuentes en su estado puramente crudo para diagnosticar cómo vienen estructuradas desde los sistemas de origen.

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# --- CONFIGURACIÓN DE FORMATO DE IMPRESIÓN ---
SEPARATOR = "=" * 90
SUB_LINE = "-" * 90

# 1. Crear la sesión de Spark optimizada para desarrollo local
# Usamos un formato de fechas ideal para fechas actuales
spark = SparkSession.builder \
    .appName("IrisFinancialExploration") \
    .master("local[*]") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .getOrCreate()

# Ajustar el nivel de log a WARN para mantener la consola limpia
spark.sparkContext.setLogLevel("WARN")

print("✨ ¡Motor PySpark encendido correctamente en local! ✨")

# 2. Definición de rutas a las fuentes crudas
PATH_TX = "../data/raw/transacciones.csv"
PATH_CLI = "../data/raw/clientes.json"

# 3. Lectura inicial descriptiva
df_transacciones_raw = spark.read.csv(PATH_TX, sep =";", header=True)
df_clientes_raw = spark.read.option("multiLine", True).json(PATH_CLI)

print("\n📊 CARGA EXITOSA: Fuentes cargadas en memoria para diagnóstico.")

✨ ¡Motor PySpark encendido correctamente en local! ✨

📊 CARGA EXITOSA: Fuentes cargadas en memoria para diagnóstico.


---

## Paso 1: Exploración

Antes de transformar los datos se realizará un exploratorio exhaustivo de las dos fuentes de datos proporcionadas. Se realizará una inspección y muestreo inicial, para luego realizar un diagnóstico para cada fuente.

### 🔍 Inspección de Esquemas y Muestreo Inicial

In [2]:


print(SEPARATOR)
print("🔍 ESTRUCTURA CRUDA DE CLIENTES")
print(SEPARATOR)
df_clientes_raw.printSchema()
df_clientes_raw.show(20, truncate=False)

print(SEPARATOR)
print("🔍 ESTRUCTURA CRUDA DE TRANSACCIONES")
print(SEPARATOR)

df_transacciones_raw.printSchema()
df_transacciones_raw.show(20, truncate=False)

🔍 ESTRUCTURA CRUDA DE CLIENTES
root
 |-- activo: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- contacto: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- telefono: string (nullable = true)
 |-- fecha_alta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- numero_documento: string (nullable = true)
 |-- productos: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- segmento: string (nullable = true)
 |-- tipo_documento: string (nullable = true)

+------+------------+------------------------------------------------+-----------+----------+------------------+----------------+---------------------------------------+--------+--------------+
|activo|ciudad      |contacto                                        |fecha_alta |id_cliente|nombre            |numero_documento|productos                              |segmento|tipo_documento|
+------+------------+-------

### 📝 Hallazgos (Inspección de Esquemas y Muestreo Inicial)
Viendo las primeras 20 filas de cada tabla se identifican unos datos bastante sucios, por lo que se procederá a hacer un análisis exaustivo por columnas. Como elementos a resaltar identificamos:

*   **Clientes (`.json`):**
    *   *Formato JSON:* El archivo viene estructurado en formato tradicional (*"multilinea"*) y no en el formato por defecto de Spark (*"JSON Lines"*). Además se identifican datos anidados en los atributos `contacto` y `productos`.
    *   *Estado de las columnas:* En todas, menos en `id_cliente` (aparentemente) se identifica un problema de no estandarización de formato y de valores.
    *   *Posibles duplicados*: Se observan indicios de posibles filas duplicadas, pues se ven nombres repetidos y números de documento nulos. Es necesario asegurar que los valores de `id_cliente` y de `numero_documento` sean únicos.
*   **Transacciones (`.csv`):** 
    *   *Separador detectado:* Haciendo una inspección rápida del csv antes de cargar los datos se identificó que se usa `;` como separador.
    *   *Estado de las columnas:* Similar a como pasa en las transacciones, se observa un problema de no estandarización de formato y valores en todas las columnas menos `id_transaccion` e `id_cliente`.
    *   *Control de valores:* Es necesario comprobar que el `id_transaccion` sea único y que los valores de `id_cliente` esten asociados a un cliente registrado.

### 👥 Diagnóstico de los datos de Clientes

In [3]:
print(SEPARATOR)
print("📊 REPORTE DE DIAGNÓSTICO: FRECUENCIAS Y VARIACIONES")
print(SEPARATOR)

# 1. Estado de Actividad
print(f"\n🔹 1. VARIACIONES HETEROGÉNEAS EN COLUMNA 'ACTIVO'\n{SUB_LINE}")
df_clientes_raw.groupBy("activo").count().sort(F.desc("count")).show(truncate=False)

# 2. Ubicación Geográfica
print(f"\n🔹 2. FRECUENCIA DE VALORES EN 'CIUDAD'\n{SUB_LINE}")
df_clientes_raw.groupBy("ciudad").count().sort(F.desc("count")).show(truncate=False)

# 3. Portafolio de Productos (Manejo de Arrays con Explode)
print(f"\n🔹 3. VALORES ÚNICOS DETECTADOS EN ARRAY 'PRODUCTOS' Y SU FRECUENCIA\n{SUB_LINE}")
df_productos_unificados = df_clientes_raw.select(F.explode_outer("productos").alias("producto_individual"))
df_productos_unificados.groupBy("producto_individual").count().sort(F.desc("count")).show(truncate=False)

# 4. Segmentación Comercial
print(f"\n🔹 4. FRECUENCIA DE VALORES EN 'SEGMENTO'\n{SUB_LINE}")
df_clientes_raw.groupBy("segmento").count().sort(F.desc("count")).show(truncate=False)

# 5. Identificación y Documentos
print(f"\n🔹 5. POSIBLES VALORES EN 'TIPO_DOCUMENTO'\n{SUB_LINE}")
df_clientes_raw.groupBy("tipo_documento").count().sort(F.desc("count")).show(truncate=False)

print(SEPARATOR)

📊 REPORTE DE DIAGNÓSTICO: FRECUENCIAS Y VARIACIONES

🔹 1. VARIACIONES HETEROGÉNEAS EN COLUMNA 'ACTIVO'
------------------------------------------------------------------------------------------
+------+-----+
|activo|count|
+------+-----+
|true  |11   |
|false |9    |
|SI    |8    |
|Si    |7    |
|NO    |6    |
|No    |5    |
|0     |4    |
|1     |2    |
+------+-----+


🔹 2. FRECUENCIA DE VALORES EN 'CIUDAD'
------------------------------------------------------------------------------------------
+------------+-----+
|ciudad      |count|
+------------+-----+
|cali        |8    |
|  Bogotá    |6    |
|medellín    |6    |
|b/quilla    |5    |
|Bogota      |5    |
|Pereira     |4    |
|Cali        |4    |
| Cúcuta     |4    |
|BOGOTA      |3    |
|Bucaramanga |2    |
|Barranquilla|2    |
|Manizales   |1    |
|Medellin    |1    |
|Cartagena   |1    |
+------------+-----+


🔹 3. VALORES ÚNICOS DETECTADOS EN ARRAY 'PRODUCTOS' Y SU FRECUENCIA
----------------------------------------------

In [4]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE INTEGRIDAD: IDENTIFICADORES Y DUPLICADOS")
print(SEPARATOR)

# ==========================================================================
# PASO 1: Validación de Calidad en id_cliente
# ==========================================================================
print(f"\n🔹 1. VALIDACIÓN DE 'id_cliente' y 'numero_documento'\n{SUB_LINE}")

df_clientes_raw.select(
    # Auditoría de IDs
    F.count(F.when(F.col("id_cliente").isNull() | (F.col("id_cliente") == ""), 1)).alias("IDs_Nulos_o_Vacios"),
    F.count(F.when(~F.col("id_cliente").rlike(r"^\d+$"), 1)).alias("IDs_No_Numericos"),
    
    # Auditoria de numeros de documento
    F.count(F.when(F.col("numero_documento").isNull() | (F.col("numero_documento") == ""), 1)).alias("Documentos_Nulos_o_Vacios")
).show()

# ==========================================================================
# PASO 2: Detección de Duplicados
# ==========================================================================
print(f"\n🔹 2. ANÁLISIS DE DUPLICADOS POR 'id_cliente'\n{SUB_LINE}")
# Agrupamos por ID y filtramos los que aparezcan más de una vez
df_duplicados_id = df_clientes_raw.groupBy("id_cliente").count().filter(F.col("count") > 1)

if df_duplicados_id.count() > 0:
    print(f"Cantidad de id_cliente repetidos: {df_duplicados_id.count()}")
    df_duplicados_id.show(truncate=False)
else:
    print("✨ ¡Perfecto! No se encontraron IDs duplicados en el archivo de clientes.")

print(f"\n🔹 3. ANÁLISIS DE DUPLICADOS POR 'numero_documento'\n{SUB_LINE}")
# Agrupamos por documento, ignorando temporalmente los nulos para no sesgar el conteo de repetidos
df_duplicados_doc = df_clientes_raw.filter(F.col("numero_documento").isNotNull() & (F.col("numero_documento") != ""))\
    .groupBy("numero_documento").count().filter(F.col("count") > 1)

if df_duplicados_doc.count() > 0:
    print(f"Cantidad de numero_documento repetidos: {df_duplicados_doc.count()}")
    df_duplicados_doc.show(truncate=False)
else:
    print("✨ ¡Perfecto! No se encontraron números de documento duplicados en el archivo de clientes.")


print(SEPARATOR)

🔍 AUDITORÍA DE INTEGRIDAD: IDENTIFICADORES Y DUPLICADOS

🔹 1. VALIDACIÓN DE 'id_cliente' y 'numero_documento'
------------------------------------------------------------------------------------------
+------------------+----------------+-------------------------+
|IDs_Nulos_o_Vacios|IDs_No_Numericos|Documentos_Nulos_o_Vacios|
+------------------+----------------+-------------------------+
|                 0|               0|                        5|
+------------------+----------------+-------------------------+


🔹 2. ANÁLISIS DE DUPLICADOS POR 'id_cliente'
------------------------------------------------------------------------------------------
Cantidad de id_cliente repetidos: 2
+----------+-----+
|id_cliente|count|
+----------+-----+
|1004      |2    |
|1011      |2    |
+----------+-----+


🔹 3. ANÁLISIS DE DUPLICADOS POR 'numero_documento'
------------------------------------------------------------------------------------------
Cantidad de numero_documento repetidos: 2
+----

### 📝 Hallazgos (Diagnóstico de los datos de Clientes)
*   En las columnas `activo`, `ciudad`, `segmento` y `tipo_documento`, las cuales funcionan como variables categóricas, es necesario estandarizar los valores.
*   Se identificaron filas con `id_cliente` y `numero_documento` repetidos, por lo que se debe explorar en detalle el registro completo para concluir si están duplicados.
*   Hay valores nulos en `numero_documento`, por lo que es necesario analizar en profundidad estos registros, para descartar que no estén duplicados, aunque tengan distinto `id_cliente`.
*   En los valores de `contacto.telefono`, `fecha_alta`, `nombre` y `numero_documento`, se encuentran disntintos formatos de escritura, por lo que se deben estandarizar.


### 💸 Diagnóstico de los datos de Transacciones

In [5]:
print(SEPARATOR)
print("📊 REPORTE DE DIAGNÓSTICO TRANSACCIONES: FRECUENCIAS Y VARIACIONES")
print(SEPARATOR)

# 1. Tipo de Producto
print(f"\n🔹 1. VARIACIONES EN COLUMNA 'tipo_producto'\n{SUB_LINE}")
df_transacciones_raw.groupBy("tipo_producto").count().sort(F.desc("count")).show(truncate=False)

# 2. Moneda de la Operación
print(f"\n🔹 2. FRECUENCIA DE VALORES EN 'moneda'\n{SUB_LINE}")
df_transacciones_raw.groupBy("moneda").count().sort(F.desc("count")).show(truncate=False)

# 3. Canal de la Transacción
print(f"\n🔹 3. FRECUENCIA DE VALORES EN 'canal'\n{SUB_LINE}")
df_transacciones_raw.groupBy("canal").count().sort(F.desc("count")).show(truncate=False)

# 4. Estado de la Transacción
print(f"\n🔹 4. FRECUENCIA DE VALORES EN 'estado'\n{SUB_LINE}")
df_transacciones_raw.groupBy("estado").count().sort(F.desc("count")).show(truncate=False)

# 5. Sucursal Origen
print(f"\n🔹 5. VARIACIONES EN COLUMNA 'sucursal'\n{SUB_LINE}")
df_transacciones_raw.groupBy("sucursal").count().sort(F.desc("count")).show(truncate=False)

print(SEPARATOR)

📊 REPORTE DE DIAGNÓSTICO TRANSACCIONES: FRECUENCIAS Y VARIACIONES

🔹 1. VARIACIONES EN COLUMNA 'tipo_producto'
------------------------------------------------------------------------------------------
+-----------------------+-----+
|tipo_producto          |count|
+-----------------------+-----+
|Cdt                    |40   |
|Cuenta de ahorros      |35   |
|CTA_AHORROS            |33   |
|Certificado de Deposito|33   |
|crédito                |31   |
|ahorros                |30   |
|CDT                    |29   |
|Cuenta Ahorros         |26   |
|CERTIFICADO DE DEPOSITO|23   |
|AHORROS                |23   |
|CORRIENTE              |22   |
|cdt                    |21   |
|cta corriente          |18   |
|CREDITO                |17   |
|Crédito                |15   |
|Credito                |13   |
|Cuenta corriente       |11   |
|Cuenta Corriente       |9    |
+-----------------------+-----+


🔹 2. FRECUENCIA DE VALORES EN 'moneda'
-----------------------------------------------------

In [16]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE INTEGRIDAD EN TRANSACCIONES: id_transaccion")
print(SEPARATOR)

# ==========================================================================
# PASO 1: Nulos, Vacíos y Formato Estructural
# ==========================================================================
print(f"\n🔹 1. VALIDACIÓN DE CALIDAD EN 'id_transaccion'\n{SUB_LINE}")

df_transacciones_raw.select(
    F.count(F.when(F.col("id_transaccion").isNull() | (F.col("id_transaccion") == ""), 1)).alias("TX_Nulas_o_Vacias"),
    F.count(F.when(~F.col("id_transaccion").rlike(r"^\d+$"), 1)).alias("TX_No_Numericas")
).show()

# ==========================================================================
# PASO 2: Detección de Duplicados Críticos
# ==========================================================================
print(f"\n🔹 2. ANÁLISIS DE REPETIDOS POR 'id_transaccion'\n{SUB_LINE}")

# Agrupamos por identificador de transacción y filtramos los que aparezcan más de una vez
df_duplicados_tx = df_transacciones_raw.groupBy("id_transaccion").count().filter(F.col("count") > 1)

print(f"Cantidad de id_transaccion duplicados: {df_duplicados_tx.count()}")

if df_duplicados_tx.count() > 0:
    print("\nMuestra de transacciones repetidas detectadas:")
    df_duplicados_tx.sort(F.desc("count")).show(10, truncate=False)
else:
    print("✨ ¡Perfecto! No se encontraron llaves duplicadas en el archivo de transacciones.")

print(SEPARATOR)

🔍 AUDITORÍA DE INTEGRIDAD EN TRANSACCIONES: id_transaccion

🔹 1. VALIDACIÓN DE CALIDAD EN 'id_transaccion'
--------------------------------------------------------------------------------
+-----------------+---------------+
|TX_Nulas_o_Vacias|TX_No_Numericas|
+-----------------+---------------+
|                0|              0|
+-----------------+---------------+


🔹 2. ANÁLISIS DE REPETIDOS POR 'id_transaccion'
--------------------------------------------------------------------------------
Cantidad de id_transaccion duplicados: 6

Muestra de transacciones repetidas detectadas:
+--------------+-----+
|id_transaccion|count|
+--------------+-----+
|500148        |2    |
|500363        |2    |
|500153        |2    |
|500060        |2    |
|500123        |2    |
|500260        |2    |
+--------------+-----+



In [ ]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE CALIDAD: 'id_cliente' EN EL ARCHIVO DE TRANSACCIONES")
print(SEPARATOR)

# Evaluar la estructura del id_cliente en transacciones antes de limpiar
df_transacciones_raw.select(
    F.count(F.when(F.col("id_cliente").isNull() | (F.col("id_cliente") == ""), 1)).alias("IDs_Nulos_o_Vacios"),
    F.count(F.when(~F.col("id_cliente").rlike(r"^\d+$"), 1)).alias("IDs_No_Numericos")
).show()


# Muestra física de los IDs que están sucios para guardarla en el reporte
print("Muestra de registros con id_cliente mal formateados en Transacciones:")
df_transacciones_raw.filter(
    F.col("id_cliente").isNotNull() & 
    (F.col("id_cliente") != "") & 
    (~F.col("id_cliente").rlike(r"^\d+$"))
).select("id_transaccion", "id_cliente").show(5, truncate=False)

print(SEPARATOR)

🔍 AUDITORÍA DE CALIDAD: 'id_cliente' EN EL ARCHIVO DE TRANSACCIONES
+------------------+----------------+
|IDs_Nulos_o_Vacios|IDs_No_Numericos|
+------------------+----------------+
|                 0|              63|
+------------------+----------------+

Muestra de registros con id_cliente mal formateados en Transacciones:
+--------------+----------+
|id_transaccion|id_cliente|
+--------------+----------+
|500379        | 1033     |
|500347        | 1033     |
|500324        | 1025     |
|500340        | 1017     |
|500023        | 1013     |
+--------------+----------+
only showing top 5 rows


In [13]:
print(SEPARATOR)
print("🔗 EVALUACIÓN DE INTEGRIDAD REFERENCIAL (APLICANDO TRIM)")
print(SEPARATOR)

# Hacemos el left_anti join aplicando F.trim() en AMBAS tablas para asegurar un cruce limpio
df_huerfanas_reales = df_transacciones_raw.join(
    df_clientes_raw,
    F.trim(df_transacciones_raw["id_cliente"]) == F.trim(df_clientes_raw["id_cliente"]),
    how="left_anti"
)

total_huerfanas_reales = df_huerfanas_reales.count()
print(f"\n❌ Cantidad REAL de transacciones huérfanas (tras remover espacios): {total_huerfanas_reales}")

if total_huerfanas_reales > 0:
    # Agrupar las transacciones huérfanas por el ID de cliente (limpio) y contar cuántas operaciones tiene cada uno
    df_recuento_fantasmas = df_huerfanas_reales.groupBy(
        F.trim(F.col("id_cliente")).alias("id_cliente_fantasma")
    ).count().sort(F.desc("count"))
    
    # Mostrar cuántos códigos de cliente distintos no existen en el maestro
    print(f"❌ Número de IDs de clientes únicos que NO existen en el maestro: {df_recuento_fantasmas.count()}")
    print("\nDetalle de IDs fantasmas y el volumen de transacciones que afectaría cada uno:")
    df_recuento_fantasmas.show(truncate=False)


else:
    print("\n✨ ¡Excelente! Al limpiar los espacios con trim(), el 100% de las transacciones cruzaron con éxito.")

🔗 EVALUACIÓN DE INTEGRIDAD REFERENCIAL (APLICANDO TRIM)

❌ Cantidad REAL de transacciones huérfanas (tras remover espacios): 20
❌ Número de IDs de clientes únicos que NO existen en el maestro: 3

Detalle de IDs fantasmas y el volumen de transacciones que afectaría cada uno:
+-------------------+-----+
|id_cliente_fantasma|count|
+-------------------+-----+
|7777               |13   |
|8888               |4    |
|9999               |3    |
+-------------------+-----+



### 📝 Hallazgos (Diagnóstico de los datos de Transacciones)
*   En las columnas `tipo_producto`, `moneda`, `canal`, `estado` y `sucursal`, las cuales funcionan como variables categóricas, es necesario estandarizar los valores.
*   Se identificaron filas con `id_transaccion` repetidos, por lo que se debe explorar en detalle el registro completo para concluir si están duplicados.
* Se encontraron registros con `id_cliente` fantasma, es decir, que no están registrados en los datos de clientes.
*   En los valores de `fecha`, `monto`, `tasa_interes` y `plazo_dias`, se encuentran disntintos formatos de escritura, por lo que se deben estandarizar.